# Model 2 - CNN Sequential

Notebook huấn luyện và đánh giá đầy đủ mô hình CNN tuần tự.

In [1]:
# 1. Import và cấu hình
import sys
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report

sys.path.append('../py')
from metrics import calculate_metrics

DATA_DIR = Path('../../../data/VN_trash_classification_preprocessing')
CHECKPOINT_DIR = Path('../checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Data:', DATA_DIR)

Device: cuda
Data: ..\..\..\data\VN_trash_classification_preprocessing


In [2]:
# 2. Xử lý ảnh và load dữ liệu
normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    normalize,
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

train_dataset = ImageFolder(DATA_DIR / 'train', transform=train_transform)
val_dataset = ImageFolder(DATA_DIR / 'val', transform=test_transform)
test_dataset = ImageFolder(DATA_DIR / 'test', transform=test_transform)

# Nạp ảnh bằng nhiều tiến trình để GPU không phải chờ CPU
loader_options = {'num_workers': 8, 'pin_memory': True, 'persistent_workers': True}
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options)

class_names = train_dataset.classes
print('Classes:', class_names)
print('Train:', len(train_dataset), '| Validation:', len(val_dataset), '| Test:', len(test_dataset))

Classes: ['Alu', 'Carton', 'Foam_box', 'Milk_box', 'Other', 'PET', 'Paper', 'Paper_cup', 'Plastic_cup']
Train: 9542 | Validation: 1685 | Test: 864


In [3]:
# 3. Tạo CNN Sequential
class CNNSequential(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 48, 3, padding=1),
            nn.BatchNorm2d(48),
            nn.ReLU(),
            nn.Conv2d(48, 48, 3, padding=1),
            nn.BatchNorm2d(48),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(48, 96, 3, padding=1),
            nn.BatchNorm2d(96),
            nn.ReLU(),
            nn.Conv2d(96, 96, 3, padding=1),
            nn.BatchNorm2d(96),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(96, 192, 3, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(),
            nn.Conv2d(192, 192, 3, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(192, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, number_of_classes),
        )

    def forward(self, images):
        return self.classifier(self.features(images))


model = CNNSequential(len(class_names)).to(device)
print(model)
print('Parameters:', sum(parameter.numel() for parameter in model.parameters()))

CNNSequential(
  (features): Sequential(
    (0): Conv2d(3, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(48, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (9): ReLU()
    (10): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (12): ReLU()
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Conv2d(96, 192

In [4]:
# 4. Loss, optimizer và scheduler
class_counts = Counter(train_dataset.targets)
class_weights = torch.tensor(
    [len(train_dataset) / (len(class_names) * class_counts[i]) for i in range(len(class_names))],
    dtype=torch.float32,
    device=device,
)

loss_function = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LEARNING_RATE * 0.01)

In [5]:
# 5. Train và validation
def run_epoch(data_loader, training=False):
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = loss_function(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)

    return total_loss / total_samples, total_correct / total_samples


best_val_accuracy = 0.0
best_state = None
best_epoch = 0

for epoch in range(EPOCHS):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_epoch(val_loader)
    scheduler.step()

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_epoch = epoch + 1
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

    print(
        f'Epoch {epoch + 1}/{EPOCHS} | '
        f'Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2%} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2%}'
    )

model.load_state_dict(best_state)
print('Best epoch:', best_epoch, '| Best validation accuracy:', f'{best_val_accuracy:.2%}')

Epoch 1/30 | Train Loss: 1.9059 | Train Acc: 32.70% | Val Loss: 1.7913 | Val Acc: 38.04%


Epoch 2/30 | Train Loss: 1.7510 | Train Acc: 38.83% | Val Loss: 1.6889 | Val Acc: 42.85%


Epoch 3/30 | Train Loss: 1.6674 | Train Acc: 42.33% | Val Loss: 1.6565 | Val Acc: 44.81%


Epoch 4/30 | Train Loss: 1.6107 | Train Acc: 45.03% | Val Loss: 1.6141 | Val Acc: 45.34%


Epoch 5/30 | Train Loss: 1.5636 | Train Acc: 46.77% | Val Loss: 1.5345 | Val Acc: 50.03%


...

Epoch 26/30 | Train Loss: 0.9637 | Train Acc: 68.90% | Val Loss: 1.0416 | Val Acc: 66.53%


Epoch 27/30 | Train Loss: 0.9624 | Train Acc: 68.52% | Val Loss: 1.0206 | Val Acc: 67.66%


Epoch 28/30 | Train Loss: 0.9434 | Train Acc: 69.14% | Val Loss: 1.0261 | Val Acc: 68.25%


Epoch 29/30 | Train Loss: 0.9395 | Train Acc: 69.56% | Val Loss: 1.0107 | Val Acc: 68.07%


Epoch 30/30 | Train Loss: 0.9398 | Train Acc: 69.87% | Val Loss: 1.0201 | Val Acc: 68.13%
Best epoch: 28 | Best validation accuracy: 68.25%


In [6]:
# 6. Test và đánh giá
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

metrics = calculate_metrics(true_labels, predicted_labels)

print()
print('Test Result')
print(f'Test Accuracy: {metrics["accuracy"]:.2%}')
print(f'Macro Precision: {metrics["macro_precision"]:.4f}')
print(f'Macro Recall: {metrics["macro_recall"]:.4f}')
print(f'Macro F1: {metrics["macro_f1"]:.4f}')
print(f'Weighted F1: {metrics["weighted_f1"]:.4f}')
print()
print(classification_report(true_labels, predicted_labels, target_names=class_names, digits=4, zero_division=0))


Test Result
Test Accuracy: 64.12%
Macro Precision: 0.6358
Macro Recall: 0.6330
Macro F1: 0.6320
Weighted F1: 0.6411

              precision    recall  f1-score   support

         Alu     0.6863    0.7071    0.6965        99
      Carton     0.8438    0.7500    0.7941       108
    Foam_box     0.5775    0.4713    0.5190        87
    Milk_box     0.7353    0.7426    0.7389       101
       Other     0.5000    0.5600    0.5283        50
         PET     0.5169    0.4510    0.4817       102
       Paper     0.6607    0.6981    0.6789       106
   Paper_cup     0.6800    0.6476    0.6634       105
 Plastic_cup     0.5221    0.6698    0.5868       106

    accuracy                         0.6412       864
   macro avg     0.6358    0.6330    0.6320       864
weighted avg     0.6459    0.6412    0.6411       864



In [7]:
# 7. Lưu model
torch.save(model.state_dict(), CHECKPOINT_DIR / 'cnn_sequential.pt')